# Setting up

## Dependencies

In [1]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pickle
import pandas as pd
import networkx as nx
from deap import gp
import logging
import random
import pandas as pd
import json
import os
from enum import Enum
import plotly.graph_objects as go
import plotly.express as px

logging.basicConfig(level=logging.INFO)

In [4]:
data={
    "J50_1.mm":{
        "ES":20,
        "LF":15
    },
    "J50_2.mm":{
        "ES":21,
        "LF":12
    }
}
df=pd.DataFrame(data).transpose()
df.index

Index(['J50_1.mm', 'J50_2.mm'], dtype='object')

## Definitons

In [23]:
class Config(Enum):
    serial_activity: str = "serial_activity"
    serial_mode: str = "serial_mode"
    serial_simultaneous: str = "serial_simultaneous"
    parallel_activity: str = "parallel_activity"
    parallel_mode: str = "parallel_mode"
    parallel_simultaneous: str = "parallel_simultaneous"


class SGS(Enum):
    serial: str = "serial"
    parallel: str = "parallel"


class DecisionType(Enum):
    activity_first: str = "activity_first"
    mode_first: str = "mode_first"
    simultaneous: str = "simultaneous"


class PlotBackend(Enum):
    plotly: int = 0
    matplotlib: int = 1

## Data acquisition

In [24]:
def fetch_entry(data_home: str, conf: Config = None, num_run=[0]) -> dict:
    if conf is None:
        conf = Config.serial_activity
    file_path = os.path.join(data_home, conf.value, f"{num_run}.json")
    if os.path.exists(file_path):
        return json.load(open(file_path))
    elif os.path.exists(file_path + r".gz"):
        import gzip

        return json.loads(gzip.open(file_path + r".gz", mode="rb").read())
    else:
        logging.warning(f"{file_path} doesn't exist.")


def fetch_group_entry(data_home: str, conf=None):
    if conf is None:
        conf = Config.serial_activity
    dir_path = os.path.join(data_home, conf.value)
    if not os.path.exists(dir_path):
        logging.warning(f"{dir_path} doesn't exist.")
        return
    files = os.listdir(dir_path)
    print(f"{len(files)} files found in {dir_path}.")
    group_entry = {}
    for file in files:
        if file.endswith(".json"):
            group_entry[int(os.path.splitext(os.path.basename(file))[0])] = json.load(
                open(os.path.join(dir_path, file))
            )
        elif file.endswith(".gz"):
            import gzip

            group_entry[int(os.path.splitext(os.path.basename(file))[0])] = json.loads(
                gzip.open(open(os.path.join(dir_path, file)), "rb").read()
            )
        else:
            logging.warning(f"Not supported file {file} detected.")
    return group_entry


def fetch_all_entry(data_home: str):
    if not os.path.exists(data_home):
        logging.warning(f"{data_home} doesn't exist.")
    records: dict = {}
    for conf in Config:
        dir_path = os.path.join(data_home, conf.value)
        if os.path.exists(dir_path):
            records[conf.value] = fetch_group_entry(data_home=data_home, conf=conf)

    logging.info(
        "\n".join(
            ["Summary:"] + [f"{conf}: {len(records[conf])} runs." for conf in records]
        )
    )


def get_results_on_test_set_all(data_home: str) -> pd.DataFrame:
    multiple_run_performances = []  # {configuration: [run1, run2, ...]}
    for con in Config:
        con_dir = os.path.join(data_home, con.value)
        if not os.path.exists(con_dir):
            continue
        log_files = os.listdir(con_dir)
        for single_run_log_file in log_files:
            log = json.load(open(os.path.join(con_dir, single_run_log_file), mode="r"))
            multiple_run_performances.append(
                {
                    "config": con.value,
                    "test_fitness": log["best_heuristic_validation"].get("test_fitness", -1),
                    "training_fitness": log["best_heuristic_validation"].get("fitness", -1),
                    "num_runs": int(single_run_log_file.split(".")[0]),
                }
            )
    return pd.DataFrame(multiple_run_performances)


def fetch_heuristic_rule(result_filepath: str) -> pd.DataFrame:
    return pd.read_json(result_filepath)


def get_best_rule_each_conf(heuristics_data: pd.DataFrame) -> pd.DataFrame:
    # choose the rule with best performance in current configuration
    best_manual_rules_each_conf: list = []

    for s in SGS:
        for d in DecisionType:
            sorted_heuristics_data = heuristics_data[
                (heuristics_data["sgs"] == s.value)
                & (heuristics_data["decision_type"] == d.value)
            ]
            if sorted_heuristics_data.empty:
                continue
            best_rule = sorted_heuristics_data.nsmallest(1, columns="fitness")[
                "name"
            ].to_string(index=False)
            best_rule_fitness = float(
                sorted_heuristics_data.nsmallest(1, columns="fitness")[
                    "fitness"
                ].to_string(index=False)
            )
            best_manual_rules_each_conf.append(
                {
                    "name": best_rule,
                    "sgs": s.value,
                    "decision_type": d.value,
                    "fitness": best_rule_fitness,
                }
            )
    return pd.DataFrame(best_manual_rules_each_conf)

## Ploting functions

In [25]:

def plot_training_best_fitness(
    data: dict[str, pd.DataFrame],
    mode=PlotBackend.plotly,
    validation_fitness=False,
    axes_label: str = None,
):
    """
    Chapter: generation_best
    """
    data_array = None
    label_array = None
    if isinstance(data, dict):
        data_array = list(data.values())
        label_array = list(data.keys())
    if isinstance(data, pd.DataFrame):
        data_array = [data]
        label_array = [axes_label] if axes_label is not None else ["fitness"]
    if mode == PlotBackend.plotly:
        fig = go.Figure()
        for ax_label, dataframe in zip(label_array, data_array):
            fig.add_trace(
                go.Scatter(
                    x=dataframe["gen"],
                    y=dataframe["fitness"],
                    mode="lines+markers",
                    name=ax_label,
                )
            )
            if validation_fitness:
                fig.add_trace(
                    go.Scatter(
                        x=dataframe["gen"],
                        y=dataframe["validation_fitness"],
                        mode="lines+markers",
                        name=ax_label + "(validation)",
                    )
                )
        return fig
    else:
        pass


def plot_training_fitness_filled_color(
    data: dict[str, pd.DataFrame], mode=PlotBackend.matplotlib
):
    """
    Chapter: fitness
    """
    if mode == PlotBackend.plotly:
        pass
    else:
        fig, ax = plt.subplot()
        for label, dataframe in data.items():
            sns.lineplot(data=dataframe, x="gen", y="fitness", label=label, ax=ax)
        return fig


def plot_fitness_distribution(
    data: pd.DataFrame,
    gen_start=0,
    gen_end=0,
    fig_title=None,
):
    """
    Chapter: population
    """
    # histogram
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import plotly.express as px
    import math

    # px.histogram(
    #     data_frame=single_run_pop_df[single_run_pop_df["gen"] == 0], x="fitness")
    if not gen_end > gen_start:
        logging.error(f"Parameter error. {gen_end=}should be greater than {gen_start}")
        return
    axes_titles = [
        f"Gen{gen}: min={round(data[data['gen'] == gen]['fitness'].min(), 1)},\
    max={round(data[data['gen'] == gen]['fitness'].max(), 1)},\
    avg={round(data[data['gen'] == gen]['fitness'].mean(), 1)}"
        for gen in range(gen_start, gen_end + 1)
    ]
    n_col = 4
    n_row = math.ceil((gen_end - gen_start) / 4)
    fig = make_subplots(
        rows=n_row, cols=n_col, subplot_titles=axes_titles, shared_yaxes=True
    )
    for row in range(1, n_row + 1):
        for col in range(1, n_col + 1):
            g = (row - 1) * n_col + col
            fig.append_trace(
                go.Histogram(x=data[data["gen"] == g]["fitness"]), row, col
            )
            fig.update_xaxes(title_text="Fitness", row=1, col=g)
    fig.update_layout(showlegend=False)
    fig.update_layout(height=300 * n_row, width=1400)
    fig.update_layout(title=dict(text=fig_title))
    return fig
def box_plot_runs_on_test_set(multiple_run_performances_df:pd.DataFrame, manual_rules_results: pd.DataFrame):
    fig = px.box(x="config", y="test_fitness", data_frame=multiple_run_performances_df)
    for index, row in manual_rules_results.iterrows():
        fig.add_trace(go.Scatter(x=["serial_activity", "parallel_simultaneous"], 
                                y=[row['fitness'], row['fitness']],
                                name="-".join(
                                    [row["name"],row["sgs"],row["decision_type"]]
                                    )))
    return fig

## Significant test

In [26]:
def wilcoxon_rank_sum_test(group_a:list, group_b:list)->float:
    from scipy.stats import mannwhitneyu
    _group_a = group_a
    _group_b = group_b
    if isinstance(group_a, (float, int)):
        if isinstance(group_b, list):
            _group_a = [group_a]*len(group_b)
            _group_b = group_b
        else:
            raise ValueError("group_a and group_b cannot be a single number at the same time")
    if isinstance(group_b, (float, int)):
        if isinstance(group_a, list):
            _group_a = group_a
            _group_b = [group_b]*len(group_a)
        else:
            raise ValueError("group_a and group_b cannot be a single number at the same time")
        
    statistic, p_value = mannwhitneyu(_group_a, _group_b)
    print("Statistic:", statistic)
    print("p-value:", p_value)

# Activity-fixed and mode fixed

In [22]:
# read data
MTGP_DATA_HOME="../results/20231107split_dynamic_terminals/"
MTGP_ACTIVITY_FIXED_HOME="/local/scratch/acitvity_fixed_dynamic/"
MTGP_MODE_FIXED_HOME="/local/scratch/mode_fixed_dynamic/"

In [23]:
MTGP_test_result_df=get_results_on_test_set_all(MTGP_DATA_HOME)
MTGP_ACTIVITY_FIXED_result_df=get_results_on_test_set_all(MTGP_ACTIVITY_FIXED_HOME)
MTGP_MODE_FIXED_result_df=get_results_on_test_set_all(MTGP_MODE_FIXED_HOME)

In [24]:
MTGP_test_result_df["source"]="MTGP"
MTGP_ACTIVITY_FIXED_result_df["source"]="MTGP-act-fixed"
MTGP_MODE_FIXED_result_df['source']="MTGP-mode-fixed"

In [25]:
combined_result=pd.concat([MTGP_test_result_df, MTGP_ACTIVITY_FIXED_result_df, MTGP_MODE_FIXED_result_df], ignore_index=True)

In [26]:
px.box(x="config", y="test_fitness", color="source", data_frame=combined_result)

# Split Training Set + Dynamic Terminals (20231107 Fixing CPM bugs)

In [7]:
# read data
DATA_HOME = r"../results/20231107split_dynamic_terminals/"
STDT_logs:dict={
    "serial_activity": fetch_entry(DATA_HOME, conf=Config.serial_activity, num_run=0),
    "serial_mode": fetch_entry(DATA_HOME, conf=Config.serial_mode, num_run=0),
    "serial_simultaneous": fetch_entry(DATA_HOME, conf=Config.serial_simultaneous, num_run=0),
    "parallel_activity":fetch_entry(DATA_HOME, Config.parallel_activity, 0),
    "parallel_mode":fetch_entry(DATA_HOME, Config.parallel_mode, 0),
    "parallel_simultaneous":fetch_entry(DATA_HOME, Config.parallel_simultaneous, 0)
}

## Single Run Analysis

In [150]:
fitness_df = {conf: pd.DataFrame(log["generation_best"]) for conf, log in STDT_logs.items()}
plot_training_best_fitness(fitness_df, validation_fitness=True)

In [151]:
plot_training_best_fitness(fitness_df["serial_mode"], validation_fitness=True, axes_label="serial_mode")

In [152]:
plot_training_best_fitness(fitness_df["serial_simultaneous"], validation_fitness=True, axes_label="serial_simultaneous")

In [153]:
plot_training_best_fitness(fitness_df["parallel_activity"], validation_fitness=True, axes_label="parallel_activity")

In [154]:
plot_training_best_fitness(fitness_df["parallel_mode"], validation_fitness=True, axes_label="parallel_mode")

In [155]:
plot_training_best_fitness(fitness_df["parallel_simultaneous"], validation_fitness=True, axes_label="parallel_simultaneous")

In [156]:
plot_fitness_distribution(pd.DataFrame(STDT_logs["serial_activity"]["population"]), 0, 12, "serial_mode")

## Test set performance (MMLIB 100)

In [159]:
# mannually designed rule
mannually_rule_result_df = fetch_heuristic_rule(r"Z:\code\discrete_optimization\results\20231107split_dynamic_terminals\20231107heuristics_result_MMLIB_100_dynamic.json")
mannually_rule_result_each_best_df = get_best_rule_each_conf(mannually_rule_result_df)
mannually_rule_result_each_best_df

,name,sgs,decision_type,fitness
0,LSTLFT,serial,activity_first,13.560480
1,LSTLFT,serial,mode_first,13.560480
2,LF,parallel,activity_first,22.568125
3,LF,parallel,mode_first,22.568125


In [27]:
# Multiple GP runs
test_results_df = get_results_on_test_set_all(data_home=DATA_HOME)

In [28]:
test_results_df.to_excel("test_results_df.xlsx")

In [161]:
box_plot_runs_on_test_set(test_results_df, mannually_rule_result_each_best_df)

In [201]:
wilcoxon_rank_sum_test(
    13.560480,
    test_results_df[test_results_df["config"] == "serial_activity"][
        "test_fitness"
    ].tolist()
)
wilcoxon_rank_sum_test(
    test_results_df[test_results_df["config"] == "serial_activity"][
        "test_fitness"
    ].tolist(),
    test_results_df[test_results_df["config"] == "serial_mode"][
        "test_fitness"
    ].tolist()
)
wilcoxon_rank_sum_test(
    test_results_df[test_results_df["config"] == "serial_activity"][
        "test_fitness"
    ].tolist(),
    test_results_df[test_results_df["config"] == "serial_simultaneous"][
        "test_fitness"
    ].tolist()
)

Statistic: 840.0
p-value: 7.47159608822232e-10
Statistic: 355.0
p-value: 0.16237502241276747
Statistic: 481.0
p-value: 0.6520436224006357


## Test set performance(MMLIB PLUS)

In [21]:
mannually_rule_result_df = fetch_heuristic_rule(r"../results/20231107split_dynamic_terminals/MMLIBPLUS_test/20231108heuristics_result_MMLIB_PLUS_dynamic.json")
mannually_rule_result_each_best_df = get_best_rule_each_conf(mannually_rule_result_df)
mannually_rule_result_df

,name,sgs,decision_type,fitness
0,LS,serial,activity_first,23.389099
1,LS,serial,mode_first,23.389099
2,LF,parallel,activity_first,68.189747
3,LF,parallel,mode_first,68.189747


In [30]:
# Multiple GP runs
test_results_df = get_results_on_test_set_all(data_home="../results/20231107split_dynamic_terminals/MMLIBPLUS_test/")

KeyError: 'fitness'

In [26]:
# px.box(x='config',y='test_fitness', data_frame=test_results_df)
# sns.boxplot(x='config',y='test_fitness', data=test_results_df)
# test_results_df.boxplot(by='config', column='test_fitness')
# test_results_df["test_fitness"]=test_results_df["test_fitness"].astype(float)
box_plot_runs_on_test_set(test_results_df,mannually_rule_result_each_best_df)

# Split Training Set + Dynamic Terminals (before fixing bugs)

In [162]:
# read data
DATA_HOME = r"Z:\code\discrete_optimization\results\20231106split_dynamic_terminals"
STDT_logs:dict={
    "serial_activity": fetch_entry(DATA_HOME, conf=Config.serial_activity, num_run=0),
    "serial_mode": fetch_entry(DATA_HOME, conf=Config.serial_mode, num_run=0),
    "serial_simultaneous": fetch_entry(DATA_HOME, conf=Config.serial_simultaneous, num_run=0),
    "parallel_activity":fetch_entry(DATA_HOME, Config.parallel_activity, 0),
    "parallel_mode":fetch_entry(DATA_HOME, Config.parallel_mode, 0),
    "parallel_simultaneous":fetch_entry(DATA_HOME, Config.parallel_simultaneous, 0)
}

## Single Run Analysis

In [163]:
fitness_df = {conf: pd.DataFrame(log["generation_best"]) for conf, log in STDT_logs.items()}
plot_training_best_fitness(fitness_df, validation_fitness=True)

In [164]:
plot_training_best_fitness(fitness_df["serial_mode"], validation_fitness=True, axes_label="serial_mode")

In [165]:
plot_training_best_fitness(fitness_df["serial_simultaneous"], validation_fitness=True, axes_label="serial_simultaneous")

In [166]:
plot_training_best_fitness(fitness_df["parallel_activity"], validation_fitness=True, axes_label="parallel_activity")

In [167]:
plot_training_best_fitness(fitness_df["parallel_mode"], validation_fitness=True, axes_label="parallel_mode")

In [168]:
plot_training_best_fitness(fitness_df["parallel_simultaneous"], validation_fitness=True, axes_label="parallel_simultaneous")

In [169]:
plot_fitness_distribution(pd.DataFrame(STDT_logs["serial_activity"]["population"]), 0, 12, "serial_mode")

## Test set performance

In [170]:
# mannually designed rule
mannually_rule_result_df = fetch_heuristic_rule(r"z:\code\discrete_optimization\results\heuristics_result_MMLIB_100_dynamic__.json")
mannually_rule_result_each_best_df = get_best_rule_each_conf(mannually_rule_result_df)
mannually_rule_result_each_best_df

,name,sgs,decision_type,fitness
0,LS,serial,activity_first,13.643746
1,LS,serial,mode_first,13.643746
2,LF,parallel,activity_first,22.568125
3,LF,parallel,mode_first,22.568125


In [171]:
# Multiple GP runs
test_results_df = get_results_on_test_set_all(data_home=DATA_HOME)

In [172]:
box_plot_runs_on_test_set(test_results_df, mannually_rule_result_each_best_df)

# Constant Training Set + Dynamic Terminals

In [ ]:
# read data
DATA_HOME = r"Z:\code\discrete_optimization\results\20231106static_dynamic_terminals"
STDY_logs:dict={
    "serial_activity": fetch_entry(DATA_HOME, conf=Config.serial_activity, num_run=0),
    # "serial_mode": fetch_entry(DATA_HOME, conf=Config.serial_mode, num_run=0),
    # "serial_simultaneous": fetch_entry(DATA_HOME, conf=Config.serial_simultaneous, num_run=0),
    # "parallel_activity":fetch_entry(DATA_HOME, Config.parallel_activity, 0),
    # "parallel_mode":fetch_entry(DATA_HOME, Config.parallel_mode, 0),
    # "parallel_simultaneous":fetch_entry(DATA_HOME, Config.parallel_simultaneous, 0)
}

## Single Run Analysis

In [ ]:
STDY_fitness_df = {conf: pd.DataFrame(log["generation_best"]) for conf, log in RTDY_logs.items()}
plot_training_best_fitness(STDY_fitness_df, validation_fitness=True)

In [ ]:
plot_training_best_fitness(STDY_fitness_df["serial_mode"], validation_fitness=True, axes_label="serial_mode")

In [ ]:
plot_training_best_fitness(STDY_fitness_df["serial_simultaneous"], validation_fitness=True, axes_label="serial_simultaneous")

In [ ]:
plot_training_best_fitness(STDY_fitness_df["parallel_activity"], validation_fitness=True, axes_label="parallel_activity")

In [ ]:
plot_training_best_fitness(STDY_fitness_df["parallel_mode"], validation_fitness=True, axes_label="parallel_mode")

In [ ]:
plot_training_best_fitness(STDY_fitness_df["parallel_simultaneous"], validation_fitness=True, axes_label="parallel_simultaneous")

In [ ]:
plot_fitness_distribution(pd.DataFrame(STDY_fitness_df["serial_activity"]["population"]), 0, 12, "serial_mode")

## Test set performance

In [ ]:
# mannually designed rule
mannually_rule_result_df = fetch_heuristic_rule(r"z:\code\discrete_optimization\results\heuristics_result_MMLIB_100_dynamic__.json")
mannually_rule_result_each_best_df = get_best_rule_each_conf(mannually_rule_result_df)
mannually_rule_result_each_best_df

,name,sgs,decision_type,fitness
0,LS,serial,activity_first,13.643746
1,LS,serial,mode_first,13.643746
2,LF,parallel,activity_first,22.568125
3,LF,parallel,mode_first,22.568125


In [ ]:
# Multiple GP runs
STDY_test_results_df = get_results_on_test_set_all(data_home=DATA_HOME)

In [ ]:
box_plot_runs_on_test_set(STDY_test_results_df, mannually_rule_result_each_best_df)

In [ ]:
box_plot_runs_on_test_set(test_results_df, mannually_rule_result_each_best_df)

# Split Training Set + Static Terminals

In [186]:
# read data
DATA_HOME = r"Z:\code\discrete_optimization\results\2023.11.2reamining_dynamic"
STST_logs:dict={
    "serial_activity": fetch_entry(DATA_HOME, conf=Config.serial_activity, num_run=0),
    "serial_mode": fetch_entry(DATA_HOME, conf=Config.serial_mode, num_run=0),
    "serial_simultaneous": fetch_entry(DATA_HOME, conf=Config.serial_simultaneous, num_run=0),
    "parallel_activity":fetch_entry(DATA_HOME, Config.parallel_activity, 0),
    "parallel_mode":fetch_entry(DATA_HOME, Config.parallel_mode, 0),
    "parallel_simultaneous":fetch_entry(DATA_HOME, Config.parallel_simultaneous, 0)
}

## Single Run Analysis

In [ ]:
fitness_df = {conf: pd.DataFrame(log["generation_best"]) for conf, log in STST_logs.items()}
plot_training_best_fitness(fitness_df, validation_fitness=True)

In [ ]:
plot_training_best_fitness(fitness_df["serial_mode"], validation_fitness=True, axes_label="serial_mode")

In [ ]:
plot_training_best_fitness(fitness_df["serial_simultaneous"], validation_fitness=True, axes_label="serial_simultaneous")

In [ ]:
plot_training_best_fitness(fitness_df["parallel_activity"], validation_fitness=True, axes_label="parallel_activity")

In [ ]:
plot_training_best_fitness(fitness_df["parallel_mode"], validation_fitness=True, axes_label="parallel_mode")

In [ ]:
plot_training_best_fitness(fitness_df["parallel_simultaneous"], validation_fitness=True, axes_label="parallel_simultaneous")

In [ ]:
plot_fitness_distribution(pd.DataFrame(STST_logs["serial_activity"]["population"]), 0, 12, "serial_mode")

## Test set performance

In [ ]:
# mannually designed rule
mannually_rule_result_df = fetch_heuristic_rule(r"Z:\code\discrete_optimization\results\heuristics_result_selected_100_EFFT.json")
mannually_rule_result_each_best_df = get_best_rule_each_conf(mannually_rule_result_df)
mannually_rule_result_each_best_df

,name,sgs,decision_type,fitness
0,LS,serial,activity_first,16.744597
1,LS,serial,mode_first,16.744597
2,LF,parallel,activity_first,27.625073
3,LF,parallel,mode_first,27.625073


In [ ]:
# Multiple GP runs
test_results_df = get_results_on_test_set_all(data_home=DATA_HOME)

In [ ]:
box_plot_runs_on_test_set(test_results_df, mannually_rule_result_each_best_df)

# Split Training Set (before removing easy instances) + static Terminals

In [173]:
# read data
DATA_HOME = r"Z:\code\discrete_optimization\results\2023.10.31split"
STDT_logs:dict={
    "serial_activity": fetch_entry(DATA_HOME, conf=Config.serial_activity, num_run=0),
    "serial_mode": fetch_entry(DATA_HOME, conf=Config.serial_mode, num_run=0),
    "serial_simultaneous": fetch_entry(DATA_HOME, conf=Config.serial_simultaneous, num_run=0),
    "parallel_activity":fetch_entry(DATA_HOME, Config.parallel_activity, 0),
    "parallel_mode":fetch_entry(DATA_HOME, Config.parallel_mode, 0),
    "parallel_simultaneous":fetch_entry(DATA_HOME, Config.parallel_simultaneous, 0)
}

## Single Run Analysis

In [174]:
fitness_df = {conf: pd.DataFrame(log["generation_best"]) for conf, log in STDT_logs.items()}
plot_training_best_fitness(fitness_df, validation_fitness=True)

In [175]:
plot_training_best_fitness(fitness_df["serial_mode"], validation_fitness=True, axes_label="serial_mode")

In [176]:
plot_training_best_fitness(fitness_df["serial_simultaneous"], validation_fitness=True, axes_label="serial_simultaneous")

In [177]:
plot_training_best_fitness(fitness_df["parallel_activity"], validation_fitness=True, axes_label="parallel_activity")

In [178]:
plot_training_best_fitness(fitness_df["parallel_mode"], validation_fitness=True, axes_label="parallel_mode")

In [179]:
plot_training_best_fitness(fitness_df["parallel_simultaneous"], validation_fitness=True, axes_label="parallel_simultaneous")

In [180]:
plot_fitness_distribution(pd.DataFrame(STDT_logs["serial_activity"]["population"]), 0, 12, "serial_mode")

## Test set performance

In [184]:
# mannually designed rule
mannually_rule_result_df = fetch_heuristic_rule(r"Z:\code\discrete_optimization\results\2023.10.31split\heuristics_result_selected_100.json")
mannually_rule_result_each_best_df = get_best_rule_each_conf(mannually_rule_result_df)
mannually_rule_result_each_best_df

,name,sgs,decision_type,fitness
0,LS,serial,activity_first,18.850357
1,LS,serial,mode_first,18.850357
2,LF,parallel,activity_first,27.625073
3,LF,parallel,mode_first,27.625073


In [182]:
# Multiple GP runs
test_results_df = get_results_on_test_set_all(data_home=DATA_HOME)

In [185]:
box_plot_runs_on_test_set(test_results_df, mannually_rule_result_each_best_df)